# Dashboard (Phase 5)

This notebook provides a **read-only live dashboard** for simulation topics.

Phase 5 scope:
- Subscribe to raw simulation topics via MQTT
- Render live person markers on an anymap-ts map
- Show city-level indicators from observer snapshots
- Keep dashboard read-only (no simulation authority)

In [1]:
# Cell purpose: Import modules, load config, and define dashboard topic subscriptions.
from __future__ import annotations

import json
from collections import defaultdict, deque
from time import sleep

from IPython.display import display
import simulated_city.mqtt as mqtt
from simulated_city.config import load_config
from simulated_city.maplibre_live import LiveMapLibreMap

cfg = load_config()
sim_cfg = cfg.simulation
mqtt_cfg = cfg.mqtt

if sim_cfg is None:
    raise ValueError("Phase 5 requires a 'simulation' section in config.yaml")

base_topic = mqtt_cfg.base_topic
topic_root = f"{base_topic}/pandemic/#"
topic_trigger = f"{base_topic}/pandemic/trigger/person_state"
topic_exposure = f"{base_topic}/pandemic/observer/exposure_event"
topic_snapshot = f"{base_topic}/pandemic/observer/city_snapshot"

population_size = sim_cfg.population_size
center_lnglat = (sim_cfg.city_center_lon, sim_cfg.city_center_lat)
print(f"Dashboard config loaded. center={center_lnglat}, subscribe={topic_root}")

Dashboard config loaded. center=(12.5683, 55.6761), subscribe=simulated-city/pandemic/#


In [2]:
# Cell purpose: Create anymap-ts map and initialize dashboard state containers.
dashboard_map = LiveMapLibreMap(center=center_lnglat, zoom=13, height="620px", width="100%")
dashboard_map.add_basemap("OpenStreetMap.Mapnik")
display(dashboard_map)

geojson_layer_name = "people_points"
geojson_layer_initialized = False
person_positions: dict[str, dict[str, float | int | str]] = {}
person_state_list: list[dict[str, float | int | str]] = []
step_people: dict[int, dict[str, dict[str, float | int | str]]] = defaultdict(dict)
processed_steps: set[int] = set()
frame_completion_ratio = 0.90
min_people_for_frame = max(1, int(population_size * frame_completion_ratio))
messages_by_topic: dict[str, int] = {
    "trigger": 0,
    "exposure": 0,
    "snapshot": 0,
}
latest_city_snapshot: dict = {}
last_rendered_step = -1
incoming_messages = deque()

def marker_color(health_status: str) -> str:
    normalized_status = health_status.strip().lower()
    if normalized_status == "infected":
        return "#d32f2f"
    if normalized_status == "recovered":
        return "#2e7d32"
    return "#1565c0"

print(
    f"Frame threshold: {min_people_for_frame}/{population_size} "
    f"({int(frame_completion_ratio * 100)}%)"
)

Frame threshold: 180/200 (90%)


In [3]:
# Cell purpose: Connect dashboard MQTT client, subscribe to raw topics, and queue incoming messages.
client = mqtt.connect_mqtt(mqtt_cfg, client_id_suffix="dashboard")
print(f"Connected to MQTT broker at {mqtt_cfg.host}:{mqtt_cfg.port}")

def on_message(client_obj, userdata, msg):
    try:
        payload = json.loads(msg.payload.decode())
    except Exception as exc:
        print(f"Dashboard payload decode error: {exc}")
        return

    incoming_messages.append((msg.topic, payload))

client.on_message = on_message
client.subscribe(topic_root)
print(f"Subscribed to {topic_root}")
print("Dashboard is live. Run Trigger + Observer notebooks, then run the processing cell.")

Connected to MQTT broker at 127.0.0.1:1883
Subscribed to simulated-city/pandemic/#
Dashboard is live. Run Trigger + Observer notebooks, then run the processing cell.


In [ ]:
# Cell purpose: Build step-synchronized frames and render all persons together.
def _rebuild_person_state_list() -> None:
    person_state_list.clear()
    for person_id in sorted(person_positions.keys()):
        state = person_positions[person_id]
        person_state_list.append(
            {
                "person_id": person_id,
                "x": float(state["lon"]),
                "y": float(state["lat"]),
                "health_status": str(state["health_status"]),
                "step": int(state.get("step", 0)),
                "ts": str(state.get("ts", "")),
            }
        )

def _render_dashboard_from_list() -> None:
    global last_rendered_step, geojson_layer_initialized

    features = []
    for state in person_state_list:
        person_id = str(state["person_id"])
        lon = float(state["x"])
        lat = float(state["y"])
        status = str(state["health_status"]).strip().lower()
        step = int(state.get("step", 0))

        features.append(
            {
                "type": "Feature",
                "geometry": {"type": "Point", "coordinates": [lon, lat]},
                "properties": {
                    "person_id": person_id,
                    "health_status": status,
                    "color": marker_color(status),
                    "popup": f"{person_id} | {status}",
                    "step": step,
                },
            }
        )
        last_rendered_step = max(last_rendered_step, step)

    if not features:
        return

    geojson_payload = {"type": "FeatureCollection", "features": features}
    if not geojson_layer_initialized:
        dashboard_map.add_geojson(
            geojson_payload,
            name=geojson_layer_name,
            layer_type="circle",
            paint={
                "circle-radius": 5,
                "circle-color": ["get", "color"],
                "circle-stroke-color": "#ffffff",
                "circle-stroke-width": 1,
                "circle-opacity": 0.95,
            },
            fit_bounds=True,
        )
        geojson_layer_initialized = True
    else:
        dashboard_map.update_geojson_source(geojson_layer_name, geojson_payload)

def process_dashboard_messages(max_items: int = 2000) -> int:
    processed = 0

    while incoming_messages and processed < max_items:
        topic, payload = incoming_messages.popleft()
        processed += 1

        if topic == topic_trigger:
            messages_by_topic["trigger"] += 1
            person_id = str(payload.get("person_id", ""))
            if not person_id:
                continue
            try:
                step = int(payload.get("step", 0))
                lat = float(payload.get("lat"))
                lon = float(payload.get("lon"))
            except (TypeError, ValueError):
                continue
            status = str(payload.get("health_status", "susceptible")).strip().lower()
            ts = str(payload.get("ts", ""))

            step_people[step][person_id] = {
                "person_id": person_id,
                "lat": lat,
                "lon": lon,
                "health_status": status,
                "step": step,
                "ts": ts,
            }

        elif topic == topic_exposure:
            messages_by_topic["exposure"] += 1

        elif topic == topic_snapshot:
            messages_by_topic["snapshot"] += 1
            latest_city_snapshot.clear()
            latest_city_snapshot.update(payload)

    candidate_steps = sorted(s for s in step_people.keys() if s not in processed_steps)
    for step in candidate_steps:
        people_count = len(step_people[step])
        has_threshold = people_count >= min_people_for_frame

        if not has_threshold:
            continue

        for person_id, state in step_people[step].items():
            person_positions[person_id] = state

        _rebuild_person_state_list()
        _render_dashboard_from_list()
        processed_steps.add(step)
        del step_people[step]

    return processed

print("Processing loop started. Interrupt the cell to stop.")
try:
    while True:
        processed_now = process_dashboard_messages(max_items=2000)
        if processed_now == 0:
            sleep(0.03)
except KeyboardInterrupt:
    print("Processing loop stopped.")

Processing loop started. Interrupt the cell to stop.


In [ ]:
# Cell purpose: Inspect continuously tracked person list (id, x, y, health_status).
tracked_count = len(person_state_list)
print(f"Tracked persons in list: {tracked_count}")

if tracked_count > 0:
    for row in person_state_list[:5]:
        print(row)
else:
    print("No tracked person positions yet. Run processing cell while trigger is publishing.")

In [ ]:
# Cell purpose: Print dashboard status counters and disconnect cleanly when finished.
print(
    f"Dashboard status: trigger_messages={messages_by_topic['trigger']}, "
    f"exposure_messages={messages_by_topic['exposure']}, "
    f"snapshot_messages={messages_by_topic['snapshot']}, "
    f"tracked_people={len(person_positions)}"
)
print(f"Latest snapshot: {latest_city_snapshot if latest_city_snapshot else 'none'}")

connector = getattr(client, "_simulated_city_connector", None)
if connector is not None:
    connector.disconnect()
    print("Disconnected from MQTT broker.")
else:
    print("No connector reference found; client disconnect skipped.")